# CE541E08 — Unit 5 · Day 43 — while Loops, break, continue and Hardy-Cross
| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 5 — Introduction to MATLAB |
| **Session** | Day 43 of 45 |
| **Topics** | while loop · back-calculate n · reservoir filling · break · continue · Hardy-Cross |
---
> **Copy each MATLAB code block and run it in MATLAB Online** at [matlab.mathworks.com](https://matlab.mathworks.com).
> Read the explanation and algorithm first. Verify your output matches the expected output.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 43"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — while Loops and Loop Control in MATLAB

A `while` loop repeats as long as a condition is true — used when you do not know in advance how many iterations are needed:

```matlab
while condition
    % body
end
```

Two loop control statements:
- **`continue`** — skip the rest of the current iteration and go to the next
- **`break`** — exit the loop immediately

These work identically to Python. The syntax is the same — `continue` and `break` are the same words.

---
## Code Block 1 — Back-calculating Manning's n by Iteration

### What this code does

We use a `while` loop to iteratively find Manning's n for a pipe where the measured velocity is known. The loop adjusts n_guess until the computed velocity matches the observed velocity to within a tolerance.

### Why each step is taken

**Analytical check first:**
We compute the exact n algebraically before the iterative solution — this gives us a target to compare against and confirms the iterative solution is working correctly.

**`n_guess = n_guess * (V_calc/V_obs)`:**
The update rule. If V_calc > V_obs, n_guess needs to be larger (higher roughness slows the flow). If V_calc < V_obs, n_guess needs to be smaller. Multiplying by the ratio converges rapidly.

**`while true ... if abs(error) < tol, break; end`:**
An infinite loop that breaks when convergence is achieved. This is safer than `while abs(error) >= tol` which can have sign issues on the first iteration.

**`if iter > 50, break; end`:**
Safety exit — prevents an infinite loop if the method fails to converge.

### Algorithm

```
1. V_obs=1.45, R=0.38, S=0.002
   n_exact = (1/V_obs)*R^(2/3)*S^0.5

2. n_guess=0.020, tol=0.000001

3. while true:
   V_calc = (1/n_guess)*R^(2/3)*S^0.5
   error  = V_obs - V_calc
   Print iteration, n_guess, V_calc, error
   if abs(error) < tol: break
   n_guess = n_guess*(V_calc/V_obs)
   iter += 1
   if iter > 50: break (safety)
```

### Expected output

```
Exact n = 0.01728
 Iter    n_guess     V_calc        Error
    0   0.020000     1.1899      0.260136
    1   0.016413     1.4500      0.000002
    2   0.016413     1.4500      0.000000
Converged: n = 0.016413 in 2 iterations
```

In [ ]:
# Copy and run this in MATLAB Online
print("""
% CE541E08 - Day 43: while Loop
% Back-calculate Manning's n by iteration

V_observed = 1.45; R = 0.38; S = 0.002;

% Analytical solution (exact answer for comparison)
n_exact = (1/V_observed) * R^(2/3) * S^0.5;
fprintf('Exact n = %.5f\n', n_exact)

% Iterative solution
n_guess = 0.020; tol = 0.000001; iter = 0;
fprintf('%5s %10s %10s %12s\n','Iter','n_guess','V_calc','Error')

while true
    V_calc = (1/n_guess) * R^(2/3) * S^0.5;
    error  = V_observed - V_calc;
    fprintf('%5d %10.6f %10.4f %12.6f\n', iter, n_guess, V_calc, error)
    if abs(error) < tol, break; end
    n_guess = n_guess * (V_calc / V_observed);
    iter    = iter + 1;
    if iter > 50, break; end
end
fprintf('Converged: n = %.6f in %d iterations\n', n_guess, iter)
""")

### 🔁 Try this

Change `n_guess = 0.020` to `n_guess = 0.050` (much worse starting guess).

- Does it still converge?
- Does it take more iterations?
- Try starting from `n_guess = 0.005` — what happens?

---
## Code Block 2 — Reservoir Filling (while Loop)

### What this code does

We simulate reservoir filling using a while loop that stops when the reservoir is full OR the inflow sequence ends — whichever comes first.

### Why each step is taken

**`while storage < capacity && day <= length(inflow)`:**
Two conditions joined by `&&` (short-circuit AND). The loop continues only if BOTH are true: reservoir is not yet full AND there are still inflow days. Either condition becoming false stops the loop.

**`if storage > capacity, storage = capacity; end`:**
Cap the storage at capacity. This prevents storage from exceeding the physical maximum while still recording that overflow occurred.

**After loop:**
Check which condition caused the exit — `storage >= capacity` means the reservoir filled; otherwise the monsoon ended first.

### Algorithm

```
1. capacity=50, storage=5, outflow=0.8
   inflow = 20-day sequence

2. while storage < capacity AND day <= len(inflow):
   storage = storage + inflow(day) - outflow
   if storage > capacity: storage = capacity
   day += 1

3. After loop:
   if storage >= capacity: print "FULL on Day X"
   else: print "Monsoon ended, final storage"
```

### Expected output

```
Reservoir FULL on Day 17
```

In [ ]:
# Copy and run this in MATLAB Online
print("""
% while loop: reservoir filling until full
capacity = 50; storage = 5; outflow = 0.8;
inflow   = [1.2,1.8,3.4,6.7,12.3,18.5,22.1,19.8,15.4,11.2,...
            8.9,6.7,5.4,4.2,3.1,2.8,2.1,1.8,1.4,1.1];
day = 1;
while storage < capacity && day <= length(inflow)
    storage = storage + inflow(day) - outflow;
    if storage > capacity, storage = capacity; end
    day = day + 1;
end
if storage >= capacity
    fprintf('Reservoir FULL on Day %d\n', day-1)
else
    fprintf('Monsoon ended. Final storage: %.2f Mm3\n', storage)
end
""")

### 🔁 Try this

Change `outflow = 0.8` to `outflow = 5.0` (very high release rate).

- Does the reservoir fill at all?
- What is the final storage at the end of the monsoon?

---
## Code Block 3 — break and continue: Sensor Data Processing

### What this code does

We process a streamflow sensor record that contains missing values (-999), spikes (>5000), and a flood peak that triggers an alert and stops scanning. We use `continue` to skip bad records and `break` to stop at the flood peak.

### Why each step is taken

**`if Q == -999 ... continue`:**
Skip the rest of this iteration and move to the next record. The missing record is flagged but not counted in statistics.

**`if Q > 5000 ... continue`:**
Same pattern — physically impossible spike is skipped.

**`if Q > 2000 ... break`:**
Stop processing when the flood peak is detected. Records after this are not needed — the alert has been issued.

**Processing order:**
Check for bad values first (before accumulating) — this prevents corrupted values from affecting the statistics.

### Algorithm

```
1. streamflow = array with -999, spikes, and valid values

2. for i = 1:length:
   Q = streamflow(i)
   if Q == -999: print MISSING; continue
   if Q > 5000:  print SPIKE; continue
   Accumulate: valid_total, valid_count, peak_flow
   if Q > 2000:  print FLOOD PEAK; break

3. Print summary: valid count, mean, peak
```

### Expected output

```
Processing sensor data:
  Record  3: MISSING - skipped
  Record  6: MISSING - skipped
  Record  9: SPIKE (9999) - skipped
  Record 13: MISSING - skipped
Valid:10  Mean:371.4  Peak:890.2
```

In [ ]:
# Copy and run this in MATLAB Online
print("""
% break and continue: skip missing data (-999)
streamflow = [234.5,267.8,-999,312.4,890.2,-999,756.4,...
              543.2,9999,345.6,289.4,245.1,-999,198.7];
valid_total = 0; valid_count = 0; peak_flow = 0;
fprintf('Processing sensor data:\n')
for i = 1:length(streamflow)
    Q = streamflow(i);
    if Q == -999
        fprintf('  Record %2d: MISSING - skipped\n', i)
        continue
    end
    if Q > 5000
        fprintf('  Record %2d: SPIKE (%.0f) - skipped\n', i, Q)
        continue
    end
    valid_total = valid_total + Q;
    valid_count = valid_count + 1;
    if Q > peak_flow, peak_flow = Q; end
end
fprintf('Valid:%d  Mean:%.1f  Peak:%.1f\n',...
        valid_count, valid_total/valid_count, peak_flow)
""")

### 🔁 Try this

Add a flood peak detection that uses `break`:

After `if Q > peak_flow, peak_flow = Q; end`, add:

```matlab
if Q > 800
    fprintf('  Record %2d: FLOOD PEAK %.1f — stopping\n', i, Q)
    break
end
```

Which record triggers the stop? How does the summary change?

---
## Code Block 4 — Hardy-Cross Pipe Network Convergence

### What this code does

We apply the Hardy-Cross method to balance head losses in a two-pipe parallel network. A while loop iterates the flow correction until head losses balance to within a tolerance.

### Why each step is taken

**Initial flow split:**
Flows are initially split proportional to diameter squared (a rough approximation). The Hardy-Cross method then corrects iteratively.

**`imbal = hf1 - hf2`:**
The head loss imbalance. For a parallel network, head loss must be equal in both branches. If hf1 > hf2, too much flow is in pipe 1.

**`dQ = imbal / (2 * (hf1/Q1 + hf2/Q2))`:**
The Hardy-Cross correction formula. `dQ` is moved from pipe 1 to pipe 2. The denominator is twice the sum of `h_f/Q` for each pipe.

**`while abs(imbal) >= tol`:**
Continue iterating while the imbalance exceeds the tolerance. Typically converges in 5-10 iterations for a well-posed network.

### Algorithm

```
1. Two parallel pipes: L1=200m, D1=0.2m and L2=300m, D2=0.25m
   Q_total = 0.05 m³/s

2. Initial split: Q1=Q_total*D1^2/(D1^2+D2^2), Q2=Q_total-Q1

3. while abs(imbal) >= tol:
   hf1 = f*(L1/D1)*(V1^2/(2*g))
   hf2 = f*(L2/D2)*(V2^2/(2*g))
   imbal = hf1 - hf2
   dQ = imbal/(2*(hf1/Q1+hf2/Q2))
   Q1 = Q1 - dQ; Q2 = Q2 + dQ

4. Print final Q1, Q2, hf1, hf2
```

### Expected output

```
 Iter   Q1(L/s)   Q2(L/s)      hf1      hf2
    0    22.222    27.778   5.0341   2.2372
    1    17.897    32.103   3.2607   3.0481
    ...
CONVERGED: Q1=18.2 L/s  Q2=31.8 L/s  hf=3.20 m
```

In [ ]:
# Copy and run this in MATLAB Online
print("""
% Hardy-Cross pipe network convergence
f=0.018; g=9.81; Q_total=0.05;
L1=200; D1=0.200; L2=300; D2=0.250;

% Initial flow split proportional to D^2
Q1 = Q_total * D1^2 / (D1^2 + D2^2);
Q2 = Q_total - Q1;
tol = 0.0001; iter = 0;

fprintf('%5s %10s %10s %8s %8s\n','Iter','Q1(L/s)','Q2(L/s)','hf1','hf2')
while true
    A1=pi*(D1/2)^2; V1=Q1/A1; hf1=f*(L1/D1)*(V1^2/(2*g));
    A2=pi*(D2/2)^2; V2=Q2/A2; hf2=f*(L2/D2)*(V2^2/(2*g));
    imbal = hf1 - hf2;
    fprintf('%5d %10.3f %10.3f %8.4f %8.4f\n',iter,Q1*1000,Q2*1000,hf1,hf2)
    if abs(imbal) < tol, break; end
    dQ = imbal / (2*(hf1/Q1 + hf2/Q2));
    Q1 = Q1 - dQ;  Q2 = Q2 + dQ;
    iter = iter + 1;
    if iter > 50, break; end
end
fprintf('CONVERGED: Q1=%.1f L/s  Q2=%.1f L/s  hf=%.2f m\n',...
        Q1*1000, Q2*1000, (hf1+hf2)/2)
""")

### 🔁 Try this

Change `D2=0.250` to `D2=0.200` (both pipes the same diameter).

- Does the flow split evenly (Q1 = Q2 = 25 L/s)?
- Are the head losses equal?
- How many iterations does convergence take?

---
## Session Summary — while Loops and Loop Control

| Concept | MATLAB | Python |
|---|---|---|
| while loop | `while condition ... end` | `while condition:` |
| Infinite loop | `while true ... end` | `while True:` |
| AND condition | `cond1 && cond2` | `cond1 and cond2` |
| Skip iteration | `continue` | `continue` |
| Exit loop | `break` | `break` |
| Safety limit | `if iter > 50, break; end` | `if iter > 50: break` |
| Hardy-Cross dQ | `imbal/(2*(hf1/Q1+hf2/Q2))` | same formula |

---
## Day 43 Assignment

Write a MATLAB script that uses a `while` loop to back-calculate the pipe roughness coefficient f (Darcy friction factor) by iteration, given:
- V_obs = 2.1 m/s, D = 0.3 m, L = 250 m, h_f = 6.5 m
- Formula: `h_f = f*(L/D)*(V^2/(2*g))`
- Start from f_guess = 0.015, adjust by `f_new = f_old * (h_f_calc/h_f_obs)`, stop when |error| < 0.0001

In [ ]:
# Copy and run this in MATLAB Online
print("""
% Day43_Assignment.m
clc; clear;
V_obs=2.1; D=0.3; L=250; h_f_obs=6.5; g=9.81;

f_guess=0.015; tol=0.0001; iter=0;
fprintf('%5s %10s %10s %10s\n','Iter','f_guess','hf_calc','Error')

while true
    h_f_calc = f_guess*(L/D)*(V_obs^2/(2*g));
    err      = h_f_obs - h_f_calc;
    fprintf('%5d %10.5f %10.4f %10.6f\n',iter,f_guess,h_f_calc,err)
    if abs(err) < tol, break; end
    f_guess = f_guess * (h_f_obs/h_f_calc);
    iter    = iter+1;
    if iter > 50, break; end
end
fprintf('Converged: f = %.5f in %d iterations\n', f_guess, iter)
""")

---
- [ ] Run all MATLAB blocks in MATLAB Online — verify outputs
- [ ] Save scripts as `.m` files
- [ ] Upload: `Unit5_MATLAB/CE541E08_U5_Day43.ipynb`
- [ ] Commit: `Day 43 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*